### Recommended: stock Colab GPU runtime (skip the conda cell)
This path uses Colab's **preinstalled CUDA PyTorch** and **does not compile** the legacy Faster R-CNN `_C` extension (THC / old ATen APIs are gone on modern PyTorch). ROI/NMS use **TorchVision** fallbacks from this fork.

The next code cell is **optional** (conda / Python pin). Leave it as `pass` unless you have a specific reason not to use stock Colab.


In [ ]:
# SKIP unless you intentionally want conda-managed Python instead of stock Colab.
# The setup path below (`setup_colab.py --colab`) no longer tries to compile legacy `_C`;
# TorchVision handles ROI/NMS.
#
# If you still insist on conda, uncomment and restart the runtime afterward:
#
# import os
# os.chdir('/content')
# !pip install -q condacolab
# import condacolab
# condacolab.install()
pass


# STTran · 300 videos · Action Genome (zipped annotations + frames)

This notebook **clones your repo on Colab**, downloads **GloVe + STTran / detector weights**, merges **`annotations.zip`** and **`frames.zip`** into a valid AG tree, then runs **`run_first5_videos_all_frames.py`** with **`VIDEO_LIMIT=300`**.

On Colab, `setup_colab.py --colab` **skips the legacy Faster R-CNN CUDA extension build** (incompatible with current PyTorch) and uses **TorchVision** ROI/NMS fallbacks.

**Outputs** (under `STTran/output/`):
- `logs/first5_videos/<VIDEO>.mp4.log` — full terminal-style log (parse for graphs / stats).
- `first5_videos/<VIDEO>.mp4/` — per-frame PNGs, GIF, legends.

After the run, **`export_graphs_json.py`** writes **`graphs.json`** in each video folder for programmatic graph extraction.

Push this repo (including `colab_minimal/`) to **GIT_REPO**, then edit the **config cell** paths (`GIT_REPO`, Drive zip paths). Use a **GPU** runtime before running.


In [ ]:
# --- EDIT THESE ---
GIT_REPO = "https://github.com/TommasoAiello08/STTran.git"
GIT_BRANCH = "main"  # change if your default branch differs

# Two zips on Google Drive (paths must exist after mounting Drive):
ANNOTATIONS_ZIP = "/content/drive/MyDrive/AG_zips/annotations.zip"
FRAMES_ZIP = "/content/drive/MyDrive/AG_zips/frames.zip"

# Where to build the Action Genome tree (ephemeral local disk on Colab):
AG_ROOT = "/content/ag_data"

# STTran code + outputs
REPO_ROOT = "/content/STTran"

# How many videos to process (first N in frame_list order that appear in the chosen split)
VIDEO_LIMIT = 300
SPLIT = "test"   # "train" or "test"

# Optional: zip results back to Drive when done
OUTPUT_ZIP_ON_DRIVE = "/content/drive/MyDrive/sttran_outputs_300.zip"


In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)


In [ ]:
"""
Clone step — run this whenever you want the notebook to use the latest code from GIT_REPO.

Each time this cell executes it:
  1) Deletes REPO_ROOT entirely (if it exists)
  2) Performs a fresh shallow clone of GIT_BRANCH

So you always match remote main (after push), without stale local edits under /content/STTran.
"""

import os, shutil, subprocess, sys, time
from pathlib import Path


def run(cmd, cwd=None):
    print("+", " ".join(cmd), flush=True)
    rc = subprocess.call(cmd, cwd=cwd)
    print("--- exit code:", rc, "---", flush=True)
    if rc != 0:
        raise subprocess.CalledProcessError(rc, cmd)


def _rm_repo() -> None:
    """Remove REPO_ROOT so the next clone cannot reuse an old tree."""
    p = Path(REPO_ROOT)
    if p.exists():
        print(f"[clone] removing existing repo: {REPO_ROOT}", flush=True)
        shutil.rmtree(p, ignore_errors=True)
        time.sleep(0.2)  # Colab sometimes races on rmtree + mkdir
    # Belt-and-suspenders (handles odd permission corner cases)
    subprocess.run(["rm", "-rf", REPO_ROOT], capture_output=True)
    if Path(REPO_ROOT).exists():
        raise RuntimeError(
            f"Could not delete {REPO_ROOT!r}. Restart runtime, then re-run this cell."
        )


_rm_repo()

# Fresh shallow clone (branch first, then default branch fallback)
clone_branch = ["git", "clone", "--depth", "1", "-b", GIT_BRANCH, GIT_REPO, REPO_ROOT]
print("+", " ".join(clone_branch), flush=True)
r = subprocess.run(clone_branch, capture_output=True, text=True)
if r.returncode != 0:
    print("Branch clone failed:", r.stderr or r.stdout or "(no stderr)")
    _rm_repo()
    clone_default = ["git", "clone", "--depth", "1", GIT_REPO, REPO_ROOT]
    print("+", " ".join(clone_default), flush=True)
    r2 = subprocess.run(clone_default, capture_output=True, text=True)
    if r2.returncode != 0:
        print("Default-branch clone failed:", r2.stderr or r2.stdout or "(no stderr)")
        raise subprocess.CalledProcessError(r2.returncode, clone_default)

print(subprocess.check_output(["git", "log", "-1", "--oneline"], cwd=REPO_ROOT).decode())
os.chdir(REPO_ROOT)
print("REPO_ROOT (fresh clone):", REPO_ROOT)


In [ ]:
# Merge annotations.zip + frames.zip -> AG_ROOT/{annotations,frames}
from pathlib import Path
import importlib.util

ann = Path(ANNOTATIONS_ZIP)
frm = Path(FRAMES_ZIP)
assert ann.is_file(), f"Missing {ann}"
assert frm.is_file(), f"Missing {frm}"

spec = importlib.util.spec_from_file_location(
    "prepare_ag_zips",
    Path(REPO_ROOT) / "colab_minimal" / "prepare_ag_zips.py",
)
mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)

mod.prepare_action_genome(ann, frm, Path(AG_ROOT), clean=True)
print("AG_ROOT ready:", AG_ROOT)


In [ ]:
import os, subprocess, sys
from pathlib import Path

# Colab-friendly setup: pip extras + GloVe + weights, TorchVision fallbacks (no legacy `_C` compile).
os.environ.setdefault("CUDA_HOME", "/usr/local/cuda")
os.environ["PATH"] = os.environ.get("PATH", "") + ":/usr/local/cuda/bin"

log_path = Path(REPO_ROOT) / "setup_colab.log"
cmd = [sys.executable, "-u", "setup_colab.py", "--colab"]

print("Starting setup... (no legacy native extension build on Colab by default)", flush=True)
with open(log_path, "w", encoding="utf-8") as logf:
    proc = subprocess.Popen(
        cmd, cwd=REPO_ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
    )
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end="")
        logf.write(line)
    rc = proc.wait()

if rc == 0:
    print("\nSetup OK. Continue to env + main run cells.")
else:
    print(f"\nSetup failed (exit {rc}). Tail {log_path.name} below if needed.")
    if log_path.is_file():
        lines = log_path.read_text(encoding="utf-8", errors="replace").splitlines()
        for line in lines[-80:]:
            print(line)


In [ ]:
# Optional: print last lines of setup log
from pathlib import Path

log_path = Path(REPO_ROOT) / "setup_colab.log"
if log_path.is_file():
    lines = log_path.read_text(encoding="utf-8", errors="replace").splitlines()
    print(f"--- last {min(120, len(lines))} lines of {log_path.name} ---")
    for line in lines[-120:]:
        print(line)
else:
    print("No setup_colab.log yet — run the setup cell first.")


In [ ]:
# Quick environment probe (safe to run anytime)
import importlib, torch, torchvision

print("python:", __import__("sys").version.split()[0])
print("torch:", torch.__version__, "cuda:", torch.cuda.is_available())
print("torchvision:", torchvision.__version__)

m = importlib.import_module("fasterRCNN.lib.model.roi_layers")
print("roi_layers import: OK", m)


In [ ]:
# RETIRED: do not run manual edits to `fasterRCNN/lib` for Colab.
# Those patches were aimed at compiling legacy `_C`, which is not needed (and usually not buildable) on modern PyTorch.
pass


In [ ]:
# (duplicate retired cell — kept to avoid breaking notebook flow if you had these numbered mentally)
pass


In [ ]:
import os
os.environ["AG_DATA_PATH"] = AG_ROOT
os.environ["STTRAN_CKPT"] = str(Path(REPO_ROOT) / "ckpts" / "sttran_predcls.tar")
os.environ["SPLIT"] = SPLIT
os.environ["VIDEO_LIMIT"] = str(VIDEO_LIMIT)
os.environ["TOPK_PER_GROUP"] = "4"
os.environ["EDGE_THRESH"] = "0.0"
os.environ["VIZ_LAYOUT"] = "circular"
os.environ["VIZ_REUSE_LAYOUT"] = "1"
os.environ["MAX_RELS"] = "2000"

for k in ("AG_DATA_PATH", "VIDEO_LIMIT", "SPLIT"):
    print(k, os.environ.get(k))


In [ ]:
# Main run (~long): one folder per video under output/first5_videos/<VIDEO>.mp4/
import subprocess, sys

print("+", sys.executable, "-u", "run_first5_videos_all_frames.py", flush=True)
proc = subprocess.run(
    [sys.executable, "-u", "run_first5_videos_all_frames.py"],
    cwd=REPO_ROOT,
    capture_output=False,  # live stdout/stderr in Colab
)
print("\n--- exit code:", proc.returncode, "---", flush=True)
if proc.returncode != 0:
    print("Script failed; scroll up for the traceback / last printed lines.", flush=True)
    raise SystemExit(proc.returncode)


In [ ]:
import os
from pathlib import Path

REPO_ROOT = "/content/STTran"
print("=== DEEP INSPECTION ===\n")

# 1. Check setup_colab.log
log_path = Path(REPO_ROOT) / "setup_colab.log"
if log_path.exists():
    print(f"--- Last 50 lines of {log_path.name} ---")
    with open(log_path, "r", encoding="utf-8") as f:
        lines = f.readlines()
        for line in lines[-50:]:
            print(line.rstrip())
else:
    print(f"{log_path.name} not found. Setup might not have run.")

print("\n" + "="*40 + "\n")

# 2. Check for runtime logs in output/
output_log_dir = Path(REPO_ROOT) / "output" / "logs" / "first5_videos"
if output_log_dir.exists():
    log_files = list(output_log_dir.glob("*.log"))
    if log_files:
        for log_file in log_files:
            print(f"--- Last 50 lines of runtime log: {log_file.name} ---")
            with open(log_file, "r", encoding="utf-8") as f:
                lines = f.readlines()
                for line in lines[-50:]:
                    print(line.rstrip())
    else:
        print("No runtime log files found in output directory.")
else:
    print(f"Runtime log directory not found: {output_log_dir}")


In [ ]:
# Parse logs -> graphs.json in each video folder (for downstream analysis)
import os
from pathlib import Path
os.environ.setdefault("TOPK_SPATIAL", "4")
os.environ.setdefault("TOPK_CONTACT", "4")
run([sys.executable, "export_graphs_json.py"], cwd=REPO_ROOT)


In [ ]:
# Optional: pack output/ into one zip on Drive (logs + viz + graphs.json)
from pathlib import Path
import shutil
out_dir = Path(REPO_ROOT) / "output"
if not out_dir.is_dir():
    print("No output directory — skipping zip.")
else:
    z = Path(OUTPUT_ZIP_ON_DRIVE)
    if z.suffix.lower() != ".zip":
        z = z.with_suffix(".zip")
    base = str(z.with_suffix(""))
    shutil.make_archive(base, "zip", root_dir=str(Path(REPO_ROOT)), base_dir="output")
    print("Wrote:", z)


In [ ]:
import os
output_path = os.path.join(REPO_ROOT, 'output')
if os.path.exists(output_path):
    print(f"Contents of {output_path}:")
    for root, dirs, files in os.walk(output_path):
        level = root.replace(output_path, '').count(os.sep)
        indent = ' ' * 4 * (level)
        print(f"{indent}{os.path.basename(root)}/")
        subindent = ' ' * 4 * (level + 1)
        for f in files:
            print(f"{subindent}{f}")
else:
    print(f"Output directory not found at {output_path}")